**Navigation** : [jumeau Python](Argumentation-03-Dung-AF-Semantics-Python.ipynb)

# Argumentation abstraite de Dung — sémantiques grounded, preferred, stable (Rust / std)

> **Notebook fondationnel** (non-agentique) de la série *ArgumentAnalysis*.
> Il précède et éclaire l'aperçu de [2-formal §6](Argumentation-05-Formal-Verification-Python.ipynb)
> et l'utilisation du solveur Tweety dans [Tweety-5](../Tweety/Tweety-05-Abstract-Argumentation-Python.ipynb).

## Pourquoi ce notebook

Le notebook [2-formal §6](Argumentation-05-Formal-Verification-Python.ipynb) invoque la sémantique
*grounded* de Dung via le solveur Tweety (JVM) comme une boîte noire : on appelle
`SimpleGroundedReasoner`, on récupère une extension. Mais **que calcule exactement ce
solveur, et pourquoi existe-t-il plusieurs sémantiques concurrentes (grounded, preferred,
stable) qui ne donnent pas toujours le même verdict ?**

Ce notebook répond à cette question en **construisant les sémantiques de zéro**, en Rust
standard (bibliothèque `std` seule, aucune crate), sans JVM ni solveur externe. L'objectif n'est pas
de remplacer Tweety (moins performant, non utilisable en production sur de grands graphes) mais
de **rendre transparent** ce que le solveur fait — de façon à ce que l'appel `getModels(theory)` du
notebook 2-formal cesse d'être magique.

**Référence** : Dung, *On the Acceptability of Arguments and its Fundamental Rôle in
Nonmonotonic Reasoning, Logic Programming and n-Person Games* (Artificial Intelligence,
1995). Le formalisme y est introduit dans sa forme la plus abstraite, indépendante de
toute structure interne des arguments.

## 1. Cadre d'argumentation abstrait (Dung 1995)

Un **abstract argumentation framework** (AF) est un couple $F = \langle A, R \rangle$ où :

- $A$ est un ensemble fini d'**arguments** (des symboles opaques — on ne s'intéresse pas à leur contenu interne, seulement à qui attaque qui) ;
- $R \subseteq A \times A$ est une relation d'**attaque** : $(a, b) \in R$ se lit « $a$ attaque $b$ ».

L'abstraction est délibérée : la théorie de Dung porte sur la *structure du débat*
(graphe d'attaque), pas sur le sens des arguments. Deux débats aux graphes identiques
reçoivent le même verdict d'acceptabilité, quelle que soit leur matière. C'est cette
abstraction qui rend les sémantiques calculables et comparables.

Implémentons le cadre :

In [2]:
use std::collections::BTreeSet;

// Cadre d'argumentation abstrait de Dung : <args, attacks>.
// `args` en Vec : c'est l'ordre d'énumération utilisé par `powerset` (cellule 10) ;
// `attacks` en BTreeSet : itération triée pour le repr, appartenance directe.
#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

#[allow(dead_code)]
impl Af {
    /// Arguments qui attaquent directement x.
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

/// Liste Python triée : ['a', 'b'] (apostrophes, virgule-espace).
fn fmt_quoted_list(items: &[&str]) -> String {
    let mut v: Vec<&str> = items.to_vec();
    v.sort();
    format!("[{}]", v.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

/// Repr fidèle du Python : AF(args=['a', 'b'], attacks=[a->b, b->a]).
fn af_repr(af: &Af) -> String {
    let att: Vec<String> = af.attacks.iter().map(|(a, b)| format!("{}->{}", a, b)).collect();
    format!("AF(args={}, attacks=[{}])", fmt_quoted_list(&af.args), att.join(", "))
}

/// Repr d'un ensemble Python non vide : {'b'} (accolades, apostrophes).
fn fmt_pyset(items: &[&str]) -> String {
    let mut v: Vec<&str> = items.to_vec();
    v.sort();
    format!("{{{}}}", v.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

// Un petit exemple ludique pour démarrer : a et b s'attaquent mutuellement.
let af0 = Af { args: vec!["a", "b"], attacks: [("a", "b"), ("b", "a")].iter().copied().collect() };
println!("{}", af_repr(&af0));
let att_a: Vec<&str> = af0.attackers("a").into_iter().collect();
println!("Attaquants de a : {}", fmt_pyset(&att_a));


AF(args=['a', 'b'], attacks=[a->b, b->a])


Attaquants de a : {'b'}


## 2. De la conflictualité à l'admissibilité

Toute la théorie repose sur une chaîne de définitions emboîtées. Chaque notion prépare
la suivante.

### 2.1 Sans conflit interne

Un ensemble $S$ est **sans conflit** (*conflict-free*) si aucun de ses membres n'en
attaque un autre : $\nexists a, b \in S$ tels que $(a, b) \in R$.

C'est la condition minimale de cohérence : un ensemble qui se contredit n'est pas
une position défendable.

In [3]:
use std::collections::BTreeSet;

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

/// Le Python imprime True/False — pas le Rust.
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

/// True si aucun argument de S n'en attaque un autre dans S.
fn is_conflict_free(af: &Af, s: &BTreeSet<&str>) -> bool {
    for (a, b) in &af.attacks {
        if s.contains(a) && s.contains(b) {
            return false;
        }
    }
    true
}

let af0 = Af { args: vec!["a", "b"], attacks: [("a", "b"), ("b", "a")].iter().copied().collect() };
let both: BTreeSet<&str> = ["a", "b"].iter().copied().collect();
let seul_a: BTreeSet<&str> = ["a"].iter().copied().collect();
println!("a,b se battent -> {{a,b}} sans conflit ? {}", py_bool(is_conflict_free(&af0, &both)));
println!("{{a}} seul  -> sans conflit ? {}", py_bool(is_conflict_free(&af0, &seul_a)));


a,b se battent -> {a,b} sans conflit ? False


{a} seul  -> sans conflit ? True


### 2.2 Défait et défendu

Pour aller plus loin que la simple absence de conflit, il faut formaliser la notion de
**défense** :

- $S$ **défait** un argument $x$ (noté $S \rightsquigarrow x$) si au moins un membre de $S$
  attaque $x$ : $\exists a \in S, (a, x) \in R$.
- $S$ **défend** un argument $x$ si $S$ défait *tous* les attaquants de $x$ :
  $\forall b$ tel que $(b, x) \in R$, $S$ défait $b$.

Autrement dit : $x$ est défendu par $S$ si toute attaque contre $x$ est contrée par une
attaque venant de $S$. C'est le cœur du raisonnement argumentatif — on accepte un
argument non parce qu'il est vrai, mais parce qu'on peut le *protéger*.

In [4]:
use std::collections::BTreeSet;

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

/// True si S défait x (au moins un membre de S attaque x).
fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(*a, x)))
}

/// True si S défend x : tout attaquant de x est défait par S.
/// Un argument non attaqué (itérateur vide) est défendu — le `all` du vide est vrai.
fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

let af0 = Af { args: vec!["a", "b"], attacks: [("a", "b"), ("b", "a")].iter().copied().collect() };
let seul_a: BTreeSet<&str> = ["a"].iter().copied().collect();
println!("Dans af0, {{a}} défend-il a ? {}", py_bool(defends(&af0, &seul_a, "a")));
// Le print Python sépare ses arguments par des espaces : l'espace avant `)` fait partie de la sortie.
println!("  (a est attaqué par b ; {{a}} défait-il b ? a->b : {} )", py_bool(af0.attacks.contains(&("a", "b"))));


Dans af0, {a} défend-il a ? True


  (a est attaqué par b ; {a} défait-il b ? a->b : True )


### 2.3 Admissible

Un ensemble $S$ est **admissible** s'il est sans conflit *et* si chacun de ses membres
est défendu par $S$ :

$$S \text{ admissible} \iff S \text{ sans conflit} \;\land\; \forall a \in S, \; S \text{ défend } a$$

L'admissibilité capture l'idée d'une position **auto-cohérente et auto-défendable** :
aucune contradiction interne, et chaque membre résiste aux attaques extérieures.

In [5]:
use std::collections::BTreeSet;

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn is_conflict_free(af: &Af, s: &BTreeSet<&str>) -> bool {
    for (a, b) in &af.attacks {
        if s.contains(a) && s.contains(b) {
            return false;
        }
    }
    true
}

fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(*a, x)))
}

fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

/// True si S est admissible : sans conflit et chacun de ses membres est défendu.
fn is_admissible(af: &Af, s: &BTreeSet<&str>) -> bool {
    is_conflict_free(af, s) && s.iter().all(|a| defends(af, s, a))
}

let af0 = Af { args: vec!["a", "b"], attacks: [("a", "b"), ("b", "a")].iter().copied().collect() };
// Ordre explicite du Python : vide, {a}, {b}, {a, b}.
let cands: [&[&str]; 4] = [&[], &["a"], &["b"], &["a", "b"]];
let _ = if true {
    for cand in cands {
    let s: BTreeSet<&str> = cand.iter().copied().collect();
    // Deux rendus du vide distincts : ici '∅' (cellule 8), ailleurs '{}' (fmt, cellule 16).
    let inner = if s.is_empty() {
        "∅".to_string()
    } else {
        s.iter().copied().collect::<Vec<_>>().join(" ")
    };
    println!("{{{}}} admissible ? {}", inner, py_bool(is_admissible(&af0, &s)));
    }
};
// Evite l'execute_result () parasite : sous evcxr, une boucle for en fin de cellule est EVALUEE.


{∅} admissible ? True


{a} admissible ? True


{b} admissible ? True


{a b} admissible ? False


## 3. Les trois sémantiques

L'admissibilité définit un *spectre* d'ensembles acceptables. Les **sémantiques**
sélectionnent, parmi les ensembles admissibles, ceux qui méritent d'être appelés
« extensions » (le verdict final). Dung en définit trois principales, qui encodent des
**attitudes de raisonnement distinctes** :

| Sémantique | Attitude | Définition |
|------------|----------|------------|
| **Grounded** | *sceptique* (le strict minimum certain) | L'unique extension complète minimale ; obtenue par point fixe : on part de $\emptyset$ et on ajoute itérativement tout argument défendu. |
| **Preferred** | *crédule* (le maximum défendable) | Les extensions complètes **maximales** (par inclusion) ; il peut y en avoir plusieurs, incompatibles entre elles. |
| **Stable** | *auto-suffisante* (elle statut sur tout) | Une extension admissible qui **défait tout argument extérieur** : $\forall x \notin S, S \rightsquigarrow x$. |
| **Semi-stable** *(§ 4bis)* | *pragmatique* (statuer au maximum) | Les admissibles qui maximisent la portée $S \cup S^+$ ; **existe toujours**, et coïncide avec la stable quand celle-ci existe. |
| **Idéale** *(§ 4bis)* | *sceptique maximale* (verdict unique) | La plus grande admissible contenue dans **toutes** les preferred ; unique, avec $\text{grounded} \subseteq \text{idéale} \subseteq \bigcap \text{preferred}$. |

Ces trois sémantiques ne sont pas redondantes : elles répondent à la question
« quels arguments accepter ? » sous trois régimes de prudence différents. Le théorème
central de Dung garantit l'inclusion **grounded $\subseteq$ preferred**, mais **stable
peut ne pas exister**, et **preferred peut compter plusieurs extensions incompatibles**.

Implémentons les trois par énumération (la théorie garantit que pour un AF fini, le nombre
d'extensions est fini ; l'énumération des $2^{|A|}$ sous-ensembles reste tractable pour
les petits graphes pédagogiques).

In [6]:
use std::collections::BTreeSet;

// Définitions seules de la chaîne complète : conflict-free -> admissible ->
// powerset -> grounded / preferred / stable (+ resume). Aucune sortie ici.

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

#[allow(dead_code)]
fn is_conflict_free(af: &Af, s: &BTreeSet<&str>) -> bool {
    for (a, b) in &af.attacks {
        if s.contains(a) && s.contains(b) {
            return false;
        }
    }
    true
}

#[allow(dead_code)]
fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(*a, x)))
}

#[allow(dead_code)]
fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

#[allow(dead_code)]
fn is_admissible(af: &Af, s: &BTreeSet<&str>) -> bool {
    is_conflict_free(af, s) && s.iter().all(|a| defends(af, s, a))
}

/// Tous les sous-ensembles, dans l'ordre exact d'itertools : taille croissante,
/// puis positions en ordre lexicographique (récursion « plus petit indice restant
/// d'abord »). PAS un balayage par masque binaire : l'ordre diffère dès que le
/// Vec d'arguments n'est pas alphabétique.
#[allow(dead_code)]
fn powerset(args: &[&'static str]) -> Vec<Vec<&'static str>> {
    fn rec(args: &[&'static str], start: usize, k: usize, cur: &[&'static str], out: &mut Vec<Vec<&'static str>>) {
        if k == 0 {
            out.push(cur.to_vec());
            return;
        }
        for i in start..=(args.len() - k) {
            let mut next = cur.to_vec();
            next.push(args[i]);
            rec(args, i + 1, k - 1, &next, out);
        }
    }
    let mut out = Vec::new();
    for r in 0..=args.len() {
        rec(args, 0, r, &[], &mut out);
    }
    out
}

/// Extension grounded : point fixe à partir de l'ensemble vide. Le résultat est
/// unique (théorème de Dung) — indépendant de l'ordre d'itération.
#[allow(dead_code)]
fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut s: BTreeSet<&'static str> = BTreeSet::new();
    let mut changed = true;
    while changed {
        changed = false;
        for x in &af.args {
            if !s.contains(x) && defends(af, &s, x) {
                s.insert(x);
                changed = true;
            }
        }
    }
    s
}

/// Extensions preferred : admissibles maximaux par inclusion, DANS l'ordre du powerset.
#[allow(dead_code)]
fn preferred(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let adm: Vec<BTreeSet<&'static str>> = powerset(&af.args)
        .into_iter()
        .map(|v| v.into_iter().collect())
        .filter(|s| is_admissible(af, s))
        .collect();
    let mut out = Vec::new();
    for a in &adm {
        let maximal = !adm.iter().any(|b| a.is_subset(b) && b != a);
        if maximal {
            out.push(a.clone());
        }
    }
    out
}

/// Extensions stables : admissibles qui défont tout argument hors de l'extension.
#[allow(dead_code)]
fn stable(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let mut out = Vec::new();
    for v in powerset(&af.args) {
        let s: BTreeSet<&'static str> = v.into_iter().collect();
        if is_admissible(af, &s) && af.args.iter().all(|x| s.contains(x) || defeats(af, &s, x)) {
            out.push(s);
        }
    }
    out
}

/// Liste Python de chaînes : ['{b,d}', '{a,d}'].
#[allow(dead_code)]
fn fmt_ext_list(items: &[String]) -> String {
    format!("[{}]", items.iter().map(|e| format!("'{}'", e)).collect::<Vec<_>>().join(", "))
}

/// Affichage compact d'une extension : '{' + éléments triés joinés par ',' SANS espace + '}'.
/// (Le vide rend '{}' — à ne pas confondre avec le '∅' de la cellule 8.)
#[allow(dead_code)]
fn fmt(e: &BTreeSet<&str>) -> String {
    format!("{{{}}}", e.iter().copied().collect::<Vec<_>>().join(","))
}

/// Calcule et affiche les trois sémantiques d'un AF (rend le triplet calculé).
#[allow(dead_code)]
fn resume(af: &Af, label: &str) -> (BTreeSet<&'static str>, Vec<BTreeSet<&'static str>>, Vec<BTreeSet<&'static str>>) {
    let g = grounded(af);
    let p = preferred(af);
    let s = stable(af);
    println!("=== {} ===", label);
    // Ligne grounded : join virgule-ESPACE, '∅' si vide.
    let fmt_g = if g.is_empty() {
        "∅".to_string()
    } else {
        g.iter().copied().collect::<Vec<_>>().join(", ")
    };
    println!("  grounded  : {{{}}}   (unique, sceptique)", fmt_g);
    let pref_strs: Vec<String> = p.iter().map(fmt).collect();
    println!("  preferred : {}   ({} ext., crédule)", fmt_ext_list(&pref_strs), p.len());
    if s.is_empty() {
        println!("  stable    : AUCUNE (pas d'extension stable)");
    } else {
        let s_strs: Vec<String> = s.iter().map(fmt).collect();
        println!("  stable    : {}   ({} ext.)", fmt_ext_list(&s_strs), s.len());
    }
    println!();
    (g, p, s)
}


## 4. Le cas canonique : grounded $\neq$ preferred $\neq$ stable

Pour que l'écart entre les trois sémantiques soit **visible**, il faut un AF assez riche
pour que les trois attitudes divergent. Considérons :

$$F = \langle \{a, b, c, d\}, \; \{(a,b), (b,a), (c,c)\} \rangle$$

- **$a$ et $b$ s'attaquent mutuellement** : ni l'un ni l'autre n'est attaqué par un tiers,
  aucun n'est défendu « gratuitement ». Ce sont des **arguments flottants** — on peut en
  accepter un (si l'on prend parti) mais pas les deux (ils se contredisent).
- **$c$ s'attaque lui-même** : un argument auto-réfuté ne peut *jamais* appartenir à un
  ensemble sans conflit (il violerait la cohérence minimale).
- **$d$ est isolé** : non attaqué, il sera accepté partout (point de référence stable).

Cet AF est **non trivial** précisément parce qu'aucune sémantique ne se réduit à une
évidence : on ne peut pas « prendre tous les non-attaqués » (ça donnerait $\{d\}$ en
ignorant les flottements) ni « tout ce qui se défend » (ça mélangerait $a$ et $b$).

In [7]:
use std::collections::BTreeSet;

// AF canonique : flottement a/b + auto-attaque c + isolé d.
// Ordre du Vec args = témoin mesuré du Python (tout ordre avec b avant a convient ;
// le tri alphabétique INVERSERAIT la liste preferred rendue).

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn fmt_quoted_list(items: &[&str]) -> String {
    let mut v: Vec<&str> = items.to_vec();
    v.sort();
    format!("[{}]", v.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

fn af_repr(af: &Af) -> String {
    let att: Vec<String> = af.attacks.iter().map(|(a, b)| format!("{}->{}", a, b)).collect();
    format!("AF(args={}, attacks=[{}])", fmt_quoted_list(&af.args), att.join(", "))
}

fn is_conflict_free(af: &Af, s: &BTreeSet<&str>) -> bool {
    for (a, b) in &af.attacks {
        if s.contains(a) && s.contains(b) {
            return false;
        }
    }
    true
}

fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(*a, x)))
}

fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

fn is_admissible(af: &Af, s: &BTreeSet<&str>) -> bool {
    is_conflict_free(af, s) && s.iter().all(|a| defends(af, s, a))
}

fn powerset(args: &[&'static str]) -> Vec<Vec<&'static str>> {
    fn rec(args: &[&'static str], start: usize, k: usize, cur: &[&'static str], out: &mut Vec<Vec<&'static str>>) {
        if k == 0 {
            out.push(cur.to_vec());
            return;
        }
        for i in start..=(args.len() - k) {
            let mut next = cur.to_vec();
            next.push(args[i]);
            rec(args, i + 1, k - 1, &next, out);
        }
    }
    let mut out = Vec::new();
    for r in 0..=args.len() {
        rec(args, 0, r, &[], &mut out);
    }
    out
}

fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut s: BTreeSet<&'static str> = BTreeSet::new();
    let mut changed = true;
    while changed {
        changed = false;
        for x in &af.args {
            if !s.contains(x) && defends(af, &s, x) {
                s.insert(x);
                changed = true;
            }
        }
    }
    s
}

fn preferred(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let adm: Vec<BTreeSet<&'static str>> = powerset(&af.args)
        .into_iter()
        .map(|v| v.into_iter().collect())
        .filter(|s| is_admissible(af, s))
        .collect();
    let mut out = Vec::new();
    for a in &adm {
        let maximal = !adm.iter().any(|b| a.is_subset(b) && b != a);
        if maximal {
            out.push(a.clone());
        }
    }
    out
}

fn stable(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let mut out = Vec::new();
    for v in powerset(&af.args) {
        let s: BTreeSet<&'static str> = v.into_iter().collect();
        if is_admissible(af, &s) && af.args.iter().all(|x| s.contains(x) || defeats(af, &s, x)) {
            out.push(s);
        }
    }
    out
}

fn fmt_ext_list(items: &[String]) -> String {
    format!("[{}]", items.iter().map(|e| format!("'{}'", e)).collect::<Vec<_>>().join(", "))
}

fn fmt(e: &BTreeSet<&str>) -> String {
    format!("{{{}}}", e.iter().copied().collect::<Vec<_>>().join(","))
}

fn resume(af: &Af, label: &str) {
    let g = grounded(af);
    let p = preferred(af);
    let s = stable(af);
    println!("=== {} ===", label);
    let fmt_g = if g.is_empty() {
        "∅".to_string()
    } else {
        g.iter().copied().collect::<Vec<_>>().join(", ")
    };
    println!("  grounded  : {{{}}}   (unique, sceptique)", fmt_g);
    let pref_strs: Vec<String> = p.iter().map(fmt).collect();
    println!("  preferred : {}   ({} ext., crédule)", fmt_ext_list(&pref_strs), p.len());
    if s.is_empty() {
        println!("  stable    : AUCUNE (pas d'extension stable)");
    } else {
        let s_strs: Vec<String> = s.iter().map(fmt).collect();
        println!("  stable    : {}   ({} ext.)", fmt_ext_list(&s_strs), s.len());
    }
    println!();
}

let af1 = Af {
    args: vec!["c", "b", "a", "d"],
    attacks: [("a", "b"), ("b", "a"), ("c", "c")].iter().copied().collect(),
};
println!("{}", af_repr(&af1));
println!();
// Les valeurs g1/p1/s1 sont recalculées par chaque cellule suivante (cellules evcxr autonomes).
resume(&af1, "F = <{a,b,c,d}, {(a,b),(b,a),(c,c)}>");


AF(args=['a', 'b', 'c', 'd'], attacks=[a->b, b->a, c->c])


=== F = <{a,b,c,d}, {(a,b),(b,a),(c,c)}> ===


  grounded  : {d}   (unique, sceptique)


  preferred : ['{b,d}', '{a,d}']   (2 ext., crédule)


  stable    : AUCUNE (pas d'extension stable)


### Lecture du verdict

Les trois sémantiques donnent des résultats **différents** sur ce graphe :

| Sémantique | Résultat | Interprétation |
|------------|----------|----------------|
| **Grounded** | $\{d\}$ | Seul $d$ est *certainement* acceptable : il est le seul non attaqué. On refuse de trancher entre $a$ et $b$ (aucun n'est défendu sans prendre parti), et $c$ s'exclut de lui-même. Attitude sceptique : on n'accepte que l'incontestable. |
| **Preferred** | $\{a, d\}$ **et** $\{b, d\}$ | Deux positions maximales défendables, **incompatibles** : soit on accepte $a$ (qui défait $b$), soit $b$ (qui défait $a$). Attitude crédule : on admet chaque position auto-défendable maximale, même s'il en existe une contradictoire. |
| **Stable** | *(aucune)* | Aucune extension stable n'existe. Pourquoi ? Une extension stable devrait défaire *tout* argument extérieur. Mais $c$ n'est attaqué par personne d'autre que lui-même : aucun argument ne peut le défaire « de l'extérieur », donc aucune extension stable ne peut le statuer. Le graphe porte un argument « indécidable de l'extérieur » → la sémantique stable échoue à exister. |

C'est précisément le théorème de Dung rendu visible :
- **grounded $\subset$ preferred** ($\{d\} \subset \{a,d\}$) ;
- **preferred $\ne$ stable** (preferred existe, stable n'existe pas) ;
- la sémantique **stable n'est pas garantie d'existence** — c'est sa fragilité théorique,
  et la raison pour laquelle les solveurs pratiques (Tweety inclus) calculent en priorité
  grounded et preferred.

> *Note de portage Rust* : l'ordre des listes d'extensions (`preferred`, `semi-stable`)
> reprend celui de l'exécution Python de référence — un ordre d'itération que Python
> lui-même ne garantit pas (flottement de hachage). Côté Rust, l'ordre vient du `Vec`
> d'arguments passé à l'AF : déterministe, calé sur le témoin Python.


In [8]:
use std::collections::BTreeSet;

// Vérification programmatique : les trois sémantiques divergent bien.
// (Chaque cellule evcxr est autonome : on redéfinit la chaîne et on RECALCULE g1/p1/s1.)

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn is_conflict_free(af: &Af, s: &BTreeSet<&str>) -> bool {
    for (a, b) in &af.attacks {
        if s.contains(a) && s.contains(b) {
            return false;
        }
    }
    true
}

fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(*a, x)))
}

fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

fn is_admissible(af: &Af, s: &BTreeSet<&str>) -> bool {
    is_conflict_free(af, s) && s.iter().all(|a| defends(af, s, a))
}

fn powerset(args: &[&'static str]) -> Vec<Vec<&'static str>> {
    fn rec(args: &[&'static str], start: usize, k: usize, cur: &[&'static str], out: &mut Vec<Vec<&'static str>>) {
        if k == 0 {
            out.push(cur.to_vec());
            return;
        }
        for i in start..=(args.len() - k) {
            let mut next = cur.to_vec();
            next.push(args[i]);
            rec(args, i + 1, k - 1, &next, out);
        }
    }
    let mut out = Vec::new();
    for r in 0..=args.len() {
        rec(args, 0, r, &[], &mut out);
    }
    out
}

fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut s: BTreeSet<&'static str> = BTreeSet::new();
    let mut changed = true;
    while changed {
        changed = false;
        for x in &af.args {
            if !s.contains(x) && defends(af, &s, x) {
                s.insert(x);
                changed = true;
            }
        }
    }
    s
}

fn preferred(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let adm: Vec<BTreeSet<&'static str>> = powerset(&af.args)
        .into_iter()
        .map(|v| v.into_iter().collect())
        .filter(|s| is_admissible(af, s))
        .collect();
    let mut out = Vec::new();
    for a in &adm {
        let maximal = !adm.iter().any(|b| a.is_subset(b) && b != a);
        if maximal {
            out.push(a.clone());
        }
    }
    out
}

fn stable(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let mut out = Vec::new();
    for v in powerset(&af.args) {
        let s: BTreeSet<&'static str> = v.into_iter().collect();
        if is_admissible(af, &s) && af.args.iter().all(|x| s.contains(x) || defeats(af, &s, x)) {
            out.push(s);
        }
    }
    out
}

let af1 = Af {
    args: vec!["c", "b", "a", "d"],
    attacks: [("a", "b"), ("b", "a"), ("c", "c")].iter().copied().collect(),
};
let g1 = grounded(&af1);
let p1 = preferred(&af1);
let s1 = stable(&af1);

let expected_g: BTreeSet<&str> = ["d"].iter().copied().collect();
println!("grounded == {{d}} ? {}", py_bool(g1 == expected_g));
// Comparaison setwise : BTreeSet<BTreeSet> des deux côtés.
let pset: BTreeSet<BTreeSet<&str>> = p1.iter().cloned().collect();
let expected_p: BTreeSet<BTreeSet<&str>> = [
    ["a", "d"].iter().copied().collect::<BTreeSet<&str>>(),
    ["b", "d"].iter().copied().collect::<BTreeSet<&str>>(),
].iter().cloned().collect();
println!("preferred == {{{{a,d}},{{b,d}}}} ? {}", py_bool(pset == expected_p));
println!("stable existe ? {}", py_bool(!s1.is_empty()));
println!();
// next(iter(p1)) Python = p1[0] ; le verdict ⊊ est vrai pour les deux extensions — sans dépendance d'ordre.
let strict = p1.first().map(|e| g1.is_subset(e) && *e != g1).unwrap_or(false);
println!(">>> grounded ⊊ preferred : {}", py_bool(strict));
println!(">>> preferred existe ET stable n'existe pas (divergence) : {}", py_bool(!p1.is_empty() && s1.is_empty()));


grounded == {d} ? True


preferred == {{a,d},{b,d}} ? True


stable existe ? False


>>> grounded ⊊ preferred : True


>>> preferred existe ET stable n'existe pas (divergence) : True


## 4bis. Quand stable n'existe pas : semi-stable et idéale

Le verdict de la section 4 laisse un problème ouvert : sur `af1`, **la sémantique stable n'existe
pas** — $c$, l'auto-attaqué, ne peut être statué par aucune extension. C'est la « fragilité »
relevée plus haut, et le curriculum la rencontrera à nouveau : [Tweety-5](../Tweety/Tweety-05-Abstract-Argumentation-Python.ipynb)
la constatera sur un cycle impair, où il introduira CF2 — une réponse qui **abandonne
l'admissibilité**. Or la littérature dispose d'une réparation qui reste dans le cadre : deux
sémantiques, absentes de ce notebook jusqu'ici.

- **Semi-stable** (Caminada 2006) : parmi les ensembles **admissibles**, ceux qui maximisent la
  *portée* $S \cup S^+$ — les arguments **acceptés** ou **attaqués** par $S$. Statuer sur le
  maximum : c'est la stable « autant que possible ». Trois propriétés la désignent comme la
  réparation canonique :
  1. elle **existe toujours** pour un AF fini — le défaut constaté sur `af1` est réparé par construction ;
  2. elle **coïncide avec la stable** dès qu'une extension stable existe — rien n'est perdu ;
  3. elle reste **admissible** — elle ne quitte pas le cadre construit en section 2.

- **Idéale** (Dung, Mancarella & Toni 2007) : la plus grande extension admissible contenue dans
  **toutes** les extensions preferred. C'est le point manquant de l'axe sceptique/crédule du
  tableau de la section 3 : un verdict **unique** — comme la grounded — mais **moins timide**,
  puisqu'il inclut tout ce qui est défendable dans *chacune* des positions crédules. Elle
  vérifie l'encadrement $\text{grounded} \subseteq \text{idéale} \subseteq \bigcap \text{preferred}$.

Implémentons les deux par-dessus la machinerie des sections 2-3 (`powerset`, `is_admissible`,
`preferred`) — quelques lignes chacune, aucune dépendance nouvelle.

In [9]:
use std::collections::BTreeSet;

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn is_conflict_free(af: &Af, s: &BTreeSet<&str>) -> bool {
    for (a, b) in &af.attacks {
        if s.contains(a) && s.contains(b) {
            return false;
        }
    }
    true
}

fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(*a, x)))
}

fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

fn is_admissible(af: &Af, s: &BTreeSet<&str>) -> bool {
    is_conflict_free(af, s) && s.iter().all(|a| defends(af, s, a))
}

fn powerset(args: &[&'static str]) -> Vec<Vec<&'static str>> {
    fn rec(args: &[&'static str], start: usize, k: usize, cur: &[&'static str], out: &mut Vec<Vec<&'static str>>) {
        if k == 0 {
            out.push(cur.to_vec());
            return;
        }
        for i in start..=(args.len() - k) {
            let mut next = cur.to_vec();
            next.push(args[i]);
            rec(args, i + 1, k - 1, &next, out);
        }
    }
    let mut out = Vec::new();
    for r in 0..=args.len() {
        rec(args, 0, r, &[], &mut out);
    }
    out
}

fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut s: BTreeSet<&'static str> = BTreeSet::new();
    let mut changed = true;
    while changed {
        changed = false;
        for x in &af.args {
            if !s.contains(x) && defends(af, &s, x) {
                s.insert(x);
                changed = true;
            }
        }
    }
    s
}

fn preferred(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let adm: Vec<BTreeSet<&'static str>> = powerset(&af.args)
        .into_iter()
        .map(|v| v.into_iter().collect())
        .filter(|s| is_admissible(af, s))
        .collect();
    let mut out = Vec::new();
    for a in &adm {
        let maximal = !adm.iter().any(|b| a.is_subset(b) && b != a);
        if maximal {
            out.push(a.clone());
        }
    }
    out
}

fn stable(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let mut out = Vec::new();
    for v in powerset(&af.args) {
        let s: BTreeSet<&'static str> = v.into_iter().collect();
        if is_admissible(af, &s) && af.args.iter().all(|x| s.contains(x) || defeats(af, &s, x)) {
            out.push(s);
        }
    }
    out
}

/// Affichage compact d'un ensemble d'arguments : {a,d} (le vide rend {}).
fn fmt(e: &BTreeSet<&str>) -> String {
    format!("{{{}}}", e.iter().copied().collect::<Vec<_>>().join(","))
}

/// S ∪ S+ : la portée de S — les arguments acceptés par S ou attaqués par S.
fn range_of(af: &Af, s: &BTreeSet<&'static str>) -> BTreeSet<&'static str> {
    let mut r: BTreeSet<&'static str> = s.clone();
    for (a, b) in &af.attacks {
        if s.contains(a) {
            r.insert(b);
        }
    }
    r
}

/// Extensions semi-stables : les admissibles qui maximisent |S ∪ S+| (Caminada 2006).
fn semi_stable(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let adm: Vec<BTreeSet<&'static str>> = powerset(&af.args)
        .into_iter()
        .map(|v| v.into_iter().collect())
        .filter(|s| is_admissible(af, s))
        .collect();
    let best = adm.iter().map(|s| range_of(af, s).len()).max().unwrap_or(0);
    adm.into_iter().filter(|s| range_of(af, s).len() == best).collect()
}

fn fmt_ext_list(items: &[String]) -> String {
    format!("[{}]", items.iter().map(|e| format!("'{}'", e)).collect::<Vec<_>>().join(", "))
}

// Retour sur l'AF canonique af1 — là où la sémantique stable n'existe pas.
let af1 = Af {
    args: vec!["c", "b", "a", "d"],
    attacks: [("a", "b"), ("b", "a"), ("c", "c")].iter().copied().collect(),
};
let s1 = stable(&af1);
let p1 = preferred(&af1);
let ss1 = semi_stable(&af1);
println!("=== af1 : le défaut et sa réparation ===");
let stable_str = if s1.is_empty() {
    "AUCUNE".to_string()
} else {
    fmt_ext_list(&s1.iter().map(fmt).collect::<Vec<_>>())
};
println!("  stable      : {}", stable_str);
let ss_strs: Vec<String> = ss1.iter().map(fmt).collect();
println!("  semi-stable : {}   ({} ext.)", fmt_ext_list(&ss_strs), ss1.len());
println!();
println!("semi-stable existe là où stable n'existe pas : {}", py_bool(!ss1.is_empty() && s1.is_empty()));
let ss_set: BTreeSet<BTreeSet<&str>> = ss1.iter().cloned().collect();
let p_set: BTreeSet<BTreeSet<&str>> = p1.iter().cloned().collect();
println!("sur af1, semi-stable == preferred            : {}", py_bool(ss_set == p_set));


=== af1 : le défaut et sa réparation ===


  stable      : AUCUNE


  semi-stable : ['{b,d}', '{a,d}']   (2 ext.)


semi-stable existe là où stable n'existe pas : True


sur af1, semi-stable == preferred            : True


In [10]:
use std::collections::BTreeSet;

// Propriété 2 : quand la stable existe, la semi-stable coïncide avec elle.
// af3 : un AF acyclique (DAG d'attaque) — la stable y existe toujours.

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn is_conflict_free(af: &Af, s: &BTreeSet<&str>) -> bool {
    for (a, b) in &af.attacks {
        if s.contains(a) && s.contains(b) {
            return false;
        }
    }
    true
}

fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(*a, x)))
}

fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

fn is_admissible(af: &Af, s: &BTreeSet<&str>) -> bool {
    is_conflict_free(af, s) && s.iter().all(|a| defends(af, s, a))
}

fn powerset(args: &[&'static str]) -> Vec<Vec<&'static str>> {
    fn rec(args: &[&'static str], start: usize, k: usize, cur: &[&'static str], out: &mut Vec<Vec<&'static str>>) {
        if k == 0 {
            out.push(cur.to_vec());
            return;
        }
        for i in start..=(args.len() - k) {
            let mut next = cur.to_vec();
            next.push(args[i]);
            rec(args, i + 1, k - 1, &next, out);
        }
    }
    let mut out = Vec::new();
    for r in 0..=args.len() {
        rec(args, 0, r, &[], &mut out);
    }
    out
}

fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut s: BTreeSet<&'static str> = BTreeSet::new();
    let mut changed = true;
    while changed {
        changed = false;
        for x in &af.args {
            if !s.contains(x) && defends(af, &s, x) {
                s.insert(x);
                changed = true;
            }
        }
    }
    s
}

fn preferred(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let adm: Vec<BTreeSet<&'static str>> = powerset(&af.args)
        .into_iter()
        .map(|v| v.into_iter().collect())
        .filter(|s| is_admissible(af, s))
        .collect();
    let mut out = Vec::new();
    for a in &adm {
        let maximal = !adm.iter().any(|b| a.is_subset(b) && b != a);
        if maximal {
            out.push(a.clone());
        }
    }
    out
}

fn stable(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let mut out = Vec::new();
    for v in powerset(&af.args) {
        let s: BTreeSet<&'static str> = v.into_iter().collect();
        if is_admissible(af, &s) && af.args.iter().all(|x| s.contains(x) || defeats(af, &s, x)) {
            out.push(s);
        }
    }
    out
}

fn fmt(e: &BTreeSet<&str>) -> String {
    format!("{{{}}}", e.iter().copied().collect::<Vec<_>>().join(","))
}

fn range_of(af: &Af, s: &BTreeSet<&'static str>) -> BTreeSet<&'static str> {
    let mut r: BTreeSet<&'static str> = s.clone();
    for (a, b) in &af.attacks {
        if s.contains(a) {
            r.insert(b);
        }
    }
    r
}

fn semi_stable(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let adm: Vec<BTreeSet<&'static str>> = powerset(&af.args)
        .into_iter()
        .map(|v| v.into_iter().collect())
        .filter(|s| is_admissible(af, s))
        .collect();
    let best = adm.iter().map(|s| range_of(af, s).len()).max().unwrap_or(0);
    adm.into_iter().filter(|s| range_of(af, s).len() == best).collect()
}

fn fmt_ext_list(items: &[String]) -> String {
    format!("[{}]", items.iter().map(|e| format!("'{}'", e)).collect::<Vec<_>>().join(", "))
}

let af3 = Af { args: vec!["a", "b", "c"], attacks: [("a", "b")].iter().copied().collect() };
let s3 = stable(&af3);
let ss3 = semi_stable(&af3);
let p3 = preferred(&af3);
println!("=== af3 = AF(args={{a,b,c}}, attacks={{a->b}}) — la stable existe ===");
println!("  stable      : {}", fmt_ext_list(&s3.iter().map(fmt).collect::<Vec<_>>()));
println!("  semi-stable : {}", fmt_ext_list(&ss3.iter().map(fmt).collect::<Vec<_>>()));
println!("  preferred   : {}", fmt_ext_list(&p3.iter().map(fmt).collect::<Vec<_>>()));
println!();
// set(ss3) == set(s3) : comparaison de Vec<BTreeSet> comme ensembles.
let ss_set: BTreeSet<BTreeSet<&str>> = ss3.iter().cloned().collect();
let s_set: BTreeSet<BTreeSet<&str>> = s3.iter().cloned().collect();
println!("coïncidence semi-stable == stable (setwise) : {}", py_bool(ss_set == s_set));


=== af3 = AF(args={a,b,c}, attacks={a->b}) — la stable existe ===


  stable      : ['{a,c}']


  semi-stable : ['{a,c}']


  preferred   : ['{a,c}']


coïncidence semi-stable == stable (setwise) : True


### Lecture du verdict : la réparation en action

Là où la stable affichait « AUCUNE », la semi-stable rend $\{a,d\}$ et $\{b,d\}$ — exactement
les preferred de `af1`. Ce n'est pas un hasard : une extension qui statue sur un maximum
d'arguments est nécessairement crédule-maximale. La différence entre les deux notions se joue
sur $c$ : personne ne l'attaque « de l'extérieur » (il ne s'attaque que lui-même, et cela ne
compte pas pour la portée d'une autre extension), donc **aucune** portée ne couvre $c$ — la
semi-stable ne décide pas $c$, elle décide **tout le reste** et laisse $c$ explicitement
indécidable. C'est le compromis exact : maximiser les arguments statués **sans jamais
sacrifier l'admissibilité** — là où CF2 (Tweety-5) paiera l'existence en quittant le cadre
admissible.

*(On montre — c'est l'esprit de l'Exercice 1 — que ces ensembles de portée maximale sont
automatiquement complets.)*

### La sémantique idéale : un verdict unique sur l'axe sceptique

La preferred est crédule : elle rend **plusieurs** verdicts incompatibles. La grounded est
sceptique : elle n'ose que l'incontestable — parfois $\emptyset$, comme sur les témoins
ci-dessous. Entre les deux, la sémantique **idéale** répond à la question que pose le pipeline
agentique de la série : *quelle position unique peut tenir un système qui doit trancher sans
prendre parti entre les extensions preferred ?* La réponse : tout ce qui est défendable **dans
chacune** des positions crédules — l'admissible maximale contenue dans toutes les preferred.
Elle existe toujours, est unique, et l'encadrement
$\text{grounded} \subseteq \text{idéale} \subseteq \bigcap\text{preferred}$ se vérifie
programmatiquement — sur deux témoins où l'écart est visible de part et d'autre.

In [11]:
use std::collections::BTreeSet;

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn is_conflict_free(af: &Af, s: &BTreeSet<&str>) -> bool {
    for (a, b) in &af.attacks {
        if s.contains(a) && s.contains(b) {
            return false;
        }
    }
    true
}

fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(*a, x)))
}

fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

fn is_admissible(af: &Af, s: &BTreeSet<&str>) -> bool {
    is_conflict_free(af, s) && s.iter().all(|a| defends(af, s, a))
}

fn powerset(args: &[&'static str]) -> Vec<Vec<&'static str>> {
    fn rec(args: &[&'static str], start: usize, k: usize, cur: &[&'static str], out: &mut Vec<Vec<&'static str>>) {
        if k == 0 {
            out.push(cur.to_vec());
            return;
        }
        for i in start..=(args.len() - k) {
            let mut next = cur.to_vec();
            next.push(args[i]);
            rec(args, i + 1, k - 1, &next, out);
        }
    }
    let mut out = Vec::new();
    for r in 0..=args.len() {
        rec(args, 0, r, &[], &mut out);
    }
    out
}

fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut s: BTreeSet<&'static str> = BTreeSet::new();
    let mut changed = true;
    while changed {
        changed = false;
        for x in &af.args {
            if !s.contains(x) && defends(af, &s, x) {
                s.insert(x);
                changed = true;
            }
        }
    }
    s
}

fn preferred(af: &Af) -> Vec<BTreeSet<&'static str>> {
    let adm: Vec<BTreeSet<&'static str>> = powerset(&af.args)
        .into_iter()
        .map(|v| v.into_iter().collect())
        .filter(|s| is_admissible(af, s))
        .collect();
    let mut out = Vec::new();
    for a in &adm {
        let maximal = !adm.iter().any(|b| a.is_subset(b) && b != a);
        if maximal {
            out.push(a.clone());
        }
    }
    out
}

fn fmt(e: &BTreeSet<&str>) -> String {
    format!("{{{}}}", e.iter().copied().collect::<Vec<_>>().join(","))
}

fn fmt_ext_list(items: &[String]) -> String {
    format!("[{}]", items.iter().map(|e| format!("'{}'", e)).collect::<Vec<_>>().join(", "))
}

/// L'extension idéale : la plus grande admissible contenue dans TOUTES les preferred.
/// (Dung, Mancarella & Toni 2007.) Le `max` Python rend le PREMIER candidat de taille
/// maximale : on remplace seulement sur len strictement plus grand.
fn ideal(af: &Af) -> BTreeSet<&'static str> {
    let prefs = preferred(af);
    let mut inter: BTreeSet<&'static str> = prefs.first().cloned().unwrap_or_default();
    for e in prefs.iter().skip(1) {
        inter = inter.intersection(e).cloned().collect();
    }
    let mut best: BTreeSet<&'static str> = BTreeSet::new();
    for v in powerset(&inter.into_iter().collect::<Vec<_>>()) {
        let s: BTreeSet<&'static str> = v.into_iter().collect();
        if is_admissible(af, &s) && s.len() > best.len() {
            best = s;
        }
    }
    best
}

// Témoin 1 : a et b flottent ; x et y se défendent mutuellement contre eux.
let af_id1 = Af {
    args: vec!["y", "b", "a", "x"],
    attacks: [("a", "b"), ("b", "a"), ("a", "x"), ("y", "a"), ("b", "y"), ("x", "b")].iter().copied().collect(),
};
// Témoin 2 : a et b flottent ; w (auto-attaqué) menace z, que seuls a et b savent défaire.
let af_id2 = Af {
    args: vec!["w", "b", "a", "z"],
    attacks: [("a", "b"), ("b", "a"), ("w", "w"), ("w", "z"), ("a", "w"), ("b", "w")].iter().copied().collect(),
};

let _ = if true {
for (name, afi) in [("af_id1", &af_id1), ("af_id2", &af_id2)] {
    let g = grounded(afi);
    let i = ideal(afi);
    let prefs = preferred(afi);
    let mut inter: BTreeSet<&'static str> = prefs.first().cloned().unwrap_or_default();
    for e in prefs.iter().skip(1) {
        inter = inter.intersection(e).cloned().collect();
    }
    println!("=== {} ===", name);
    let pref_strs: Vec<String> = prefs.iter().map(fmt).collect();
    println!("  preferred   : {}", fmt_ext_list(&pref_strs));
    println!("  grounded    : {}", fmt(&g));
    println!("  idéale      : {}", fmt(&i));
    println!("  ∩ preferred : {}", fmt(&inter));
    println!("  encadrement g ⊆ i ⊆ ∩pref : {}", py_bool(g.is_subset(&i) && i.is_subset(&inter)));
    println!("  les trois ne coïncident pas : {}", py_bool(!(g == i && i == inter)));
    println!();
}
};


=== af_id1 ===


  preferred   : ['{x,y}']


  grounded    : {}


  idéale      : {x,y}


  ∩ preferred : {x,y}


  encadrement g ⊆ i ⊆ ∩pref : True


... (4 outputs left out)

  idéale      : {}


  ∩ preferred : {z}


  encadrement g ⊆ i ⊆ ∩pref : True


  les trois ne coïncident pas : True


### Lecture des deux témoins : le sandwich rendu visible

Sur `af_id1`, $x$ et $y$ se défendent **mutuellement** ($y$ défait l'attaquant de $x$, $x$
défait celui de $y$) : aucun n'est défendu à partir de $\emptyset$, donc la grounded reste
vide — mais ensemble ils sont admissibles, et présents dans l'unique extension preferred.
L'idéale accepte alors $\{x,y\}$ là où la grounded ne disait rien : **moins timide**, pour
le même prix — un verdict unique.

Sur `af_id2`, l'argument $z$ est présent dans **chaque** preferred — que l'on choisisse $a$ ou
$b$ comme parti pris, l'un des deux défait $w$, l'agresseur de $z$. Donc
$z \in \bigcap\text{preferred}$. Mais cette intersection n'est **pas** admissible : $z$ n'y
est défendu par personne ($w$ n'est défait que par $a$ ou par $b$, jamais ensemble dans une
même extension). L'idéale retombe alors sur $\emptyset$ : elle ne promet que ce qui est
défendable **sans choisir** de camp.

Les deux témoins ensemble situent exactement la sémantique : strictement au-dessus de la
grounded dès qu'existe une défense mutuelle universelle, strictement sous l'intersection dès
qu'un argument commun n'est défendu que par des partis pris. Construire un AF où les **trois**
diffèrent deux à deux est l'objet de l'Exercice 4.

## 5. Retour au solveur Tweety

Maintenant que les sémantiques sont transparentes, relisons l'appel du notebook
[2-formal §6](Argumentation-05-Formal-Verification-Python.ipynb) :

```python
reasoner_d = SimpleGroundedReasoner()
models = reasoner_d.getModels(theory)   # Collection<Extension>
```

`getModels` renvoie la collection des extensions selon la sémantique du *reasoner*. Pour
`SimpleGroundedReasoner`, c'est exactement le point fixe calculé par notre fonction
`grounded` ci-dessus — une seule extension, la plus prudente. Les *reasoners*
`SimplePreferredReasoner` et `SimpleCompleteReasoner` implémentent les autres colonnes
du tableau. Le pont JPype ne fait rien d'autre qu'appeler, en JVM, l'algorithme que nous
venons de reconstruire à la main.

**Ce que ce notebook apporte** au-delà de Tweety-5 : la compréhension de l'algorithme,
pas seulement son usage. C'est la différence entre *invoquer* un vérificateur formel
(le pipeline agentique de la série) et *comprendre* ce qu'il calcule (ce notebook).

> Pour les preuves formelles des théorèmes d'inclusion et d'existence (grounded est
> l'extension complète minimale, stable peut être vide), voir le notebook Lean
> [Tweety-5b](../Tweety/Tweety-05b-Argumentation-Lean.ipynb).

## 6. Exercices

Les quatre exercices suivants approfondissent la théorie. Ils sont laissés **incomplets**
par construction (stub) : à vous de les remplir. Le notebook s'exécute de bout en bout
même non complété — vos implémentations remplaceront les valeurs neutres
(`Option<bool>::None`, AF vide, étiquetage tout `UNDEC`).


### Exercice 1 — Complétude vs admissibilité

Une extension **complète** est admissible *et* contient tout argument qu'elle défend :
$S$ complète $\iff$ $S$ admissible et $\forall x \in A$, si $S$ défend $x$ alors $x \in S$.
La grounded est l'unique complète minimale, les preferred sont les complètes maximales.

**Objectif** : implémenter `is_complete(af, S)` et vérifier que la grounded et les
preferred de `af1` sont bien complètes (mais qu'un admissible non-maximal comme $\{a\}$
ne l'est pas — pourquoi ?).

*Indice* : un argument défendu mais absent de $S$ suffit à le disqualifier.

In [12]:
use std::collections::BTreeSet;

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}


#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

// Exercice 1 : compléter cette fonction
// True si S est complet : admissible ET contient tout argument qu'il défend.
// Etape 1 : vérifier l'admissibilité (déjà disponible : is_admissible, cellules précédentes).
// Etape 2 : pour tout x défendu par S, x doit appartenir à S.
// TODO etudiant — Option<bool>::None joue le rôle du `return None` Python.
#[allow(dead_code)]
fn is_complete(_af: &Af, _s: &BTreeSet<&str>) -> Option<bool> {
    None
}

// Test (à décommenter une fois complété — sous evcxr, redéfinir la chaîne
// is_conflict_free/defends/is_admissible et af1 dans la même cellule) :
// let g1 = grounded(&af1);
// let p1 = preferred(&af1);
// println!("grounded complet ?", is_complete(&af1, &g1).unwrap());
// println!("preferred complet ?", p1.iter().all(|e| is_complete(&af1, e) == Some(true)));
// let seul_a: BTreeSet<&str> = ["a"].iter().copied().collect();
// println!("{{a}} (admissible non-maximal) complet ?", is_complete(&af1, &seul_a).unwrap());


### Exercice 2 — Un AF où la sémantique stable existe

Sur `af1`, la sémantique stable n'existait pas (à cause de l'auto-attaque de $c$).
Construisez un AF **sans auto-attaque ni cycle impair** où la stable existe *et* coïncide
avec la preferred.

**Objectif** : définir `af2` et vérifier que `stable(af2) == preferred(af2)` (même
collection d'extensions).

*Indice* : un AF acyclique (les arguments forment un DAG d'attaque) admet toujours une
unique extension stable, qui coïncide avec la grounded et la preferred.

In [13]:
use std::collections::BTreeSet;

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}


#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

// Exercice 2 : définir un AF où stable == preferred
// La valeur neutre ci-dessous est l'AF vide — exactement le AF(set(), set()) Python.
let af2 = Af {
    args: Vec::new(),        // TODO etudiant : choisir args
    attacks: BTreeSet::new(), // TODO etudiant : choisir attacks (sans auto-attaque)
};

// Test (à décommenter une fois complété — redéfinir la chaîne + resume dans la cellule) :
// let (p2, s2) = (preferred(&af2), stable(&af2));
// let ps: BTreeSet<_> = p2.iter().cloned().collect();
// let ss: BTreeSet<_> = s2.iter().cloned().collect();
// println!("stable == preferred ?", ps == ss);
let _ = &af2; // silencieux tant que l'exercice n'est pas complété (aucune sortie attendue)


### Exercice 3 — Étiquetage IN / OUT / UNDEC

Les sémantiques admettent une formulation équivalente en **étiquetages** : chaque
argument reçoit l'étiquette IN (accepté), OUT (rejeté) ou UNDEC (indécidable), avec les
règles : (i) $x$ est OUT si un IN l'attaque ; (ii) $x$ est IN si tous ses attaquants sont
OUT. Le labeling grounded de `af1` devrait donner $d$=IN, $c$=UNDEC (auto-attaqué, mais rejeté par aucun IN), et
$a, b$=UNDEC (le flottement non tranché).

**Objectif** : implémenter `grounded_labeling(af)` renvoyant un dict `{arg: "IN"/"OUT"/"UNDEC"}`,
et vérifier qu'il est cohérent avec `grounded(af)` (IN $=$ l'extension grounded, du moins
quand aucun argument n'est UNDEC dans la grounded).

*Indice* : OUT = attaqué par un IN ; IN = non attaqué, ou tous ses attaquants OUT ;
UNDEC = ni IN ni OUT (le reste). Itérez jusqu'à stabilisation, comme pour le point fixe.

In [14]:
use std::collections::{BTreeMap, BTreeSet};

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}


#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

// Exercice 3 : compléter le labeling grounded
// Renvoie {arg: "IN"|"OUT"|"UNDEC"} selon le labeling grounded.
// Etape 1 : IN = argument dont tous les attaquants sont OUT (ou sans attaquant).
// Etape 2 : OUT = argument attaqué par au moins un IN.
// Etape 3 : itérer jusqu'au point fixe (comme grounded).
// TODO etudiant — le stub renvoie l'étiquetage tout-UNDEC du Python.
#[allow(dead_code)]
fn grounded_labeling(af: &Af) -> BTreeMap<&'static str, &'static str> {
    let mut labels: BTreeMap<&'static str, &'static str> = BTreeMap::new();
    for a in &af.args {
        labels.insert(a, "UNDEC");
    }
    labels
}

// Test (à décommenter une fois complété — redéfinir af1 + grounded dans la cellule) :
// let lab = grounded_labeling(&af1);
// println!("Labeling grounded de af1 : {:?}", lab);
// let g1 = grounded(&af1);
// let ins: BTreeSet<&str> = lab.iter().filter(|(_, l)| **l == "IN").map(|(a, _)| *a).collect();
// println!("Cohérent avec grounded={{d}} ?", ins == g1);


### Exercice 4 — L'axe sceptique complet : grounded, idéale, $\bigcap$preferred

Le § 4bis a montré deux témoins : sur `af_id1`, l'idéale dépasse la grounded
($\emptyset \subsetneq \{x,y\}$) mais égale l'intersection ; sur `af_id2`, elle égale la
grounded ($\emptyset$) mais reste sous l'intersection ($\{z\}$). Dans chacun, **deux des trois
coïncident**.

**Objectif** : construire un AF où les trois ensembles diffèrent **deux à deux** —
$\text{grounded} \subsetneq \text{idéale} \subsetneq \bigcap\text{preferred}$, strictement.

**Indices :**
- *Indice 1* : pour dépasser la grounded, il faut de la **défense mutuelle** — des arguments
  défendus seulement par d'autres arguments eux-mêmes non défendus à partir de $\emptyset$,
  mais présents dans toutes les preferred.
- *Indice 2* : pour rester sous l'intersection, il faut un argument **commun à toutes les
  preferred** mais défendu uniquement par des arguments qui changent selon l'extension.
- *Etape 1* : écrire `af4` (les deux mécanismes des témoins peuvent coexister dans un même
  AF) ; *Etape 2* : décommenter le test ; *Etape 3* : vérifier que les trois ensembles
  forment bien une chaîne strictement croissante.

In [15]:
use std::collections::BTreeSet;

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}


#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

// Exercice 4 : construire un AF où grounded, idéale et ∩preferred diffèrent DEUX À DEUX
let af4 = Af {
    args: Vec::new(),        // TODO etudiant : choisir args
    attacks: BTreeSet::new(), // TODO etudiant : choisir attacks
};

// Test (à décommenter une fois complété — redéfinir la chaîne complète dans la cellule) :
// let (g4, i4) = (grounded(&af4), ideal(&af4));
// let prefs = preferred(&af4);
// let inter4 = prefs.first().cloned().unwrap_or_default();
// println!("grounded : {} | idéale : {} | ∩preferred : {}", fmt(&g4), fmt(&i4), fmt(&inter4));
let _ = &af4; // silencieux tant que l'exercice n'est pas complété
println!("Exercice a completer : axe sceptique grounded / idéale / ∩preferred");


Exercice a completer : axe sceptique grounded / idéale / ∩preferred


## Aller plus loin — la **trajectoire** d'étiquetage

Tout ce qui précède calcule un étiquetage sur un AF **déjà complet** : tous les arguments
sont posés, toutes les attaques sont connues, on cherche le point fixe. Mais un débat réel
n'arrive pas d'un bloc — les arguments sont **énoncés l'un après l'autre**, et un argument
ne peut pas être attaqué par un argument pas encore prononcé.

Étiquetons donc après *chaque* arrivée, et regardons la **suite** d'étiquetages plutôt que
le seul étiquetage final. Deux propriétés s'opposent :

- l'étiquetage **final** ne dépend pas de l'ordre d'arrivée — Dung est *order-blind* ;
- la **trajectoire**, elle, en dépend.

La trajectoire porte donc exactement l'information que l'étiquetage statique jette : *quand*
un argument a basculé, et sous l'effet de quelle arrivée. C'est aussi le lieu où se voit la
**non-monotonie** — un argument accepté peut redevenir indécidable quand son attaquant arrive.


In [16]:
use std::collections::{BTreeMap, BTreeSet};

// Raccourci assumé : on lit les étiquettes depuis `grounded()`, déjà fourni plus haut.
// ⚠ Ce n'est PAS la solution de l'Exercice 3, qui demande la formulation par point fixe
// (itération IN/OUT jusqu'à stabilisation) — celle-là reste à écrire.

#[allow(dead_code)]
struct Af {
    args: Vec<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

#[allow(dead_code)]
impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter().filter(|(_, b)| *b == x).map(|(a, _)| *a).collect()
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(*a, x)))
}

fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut s: BTreeSet<&'static str> = BTreeSet::new();
    let mut changed = true;
    while changed {
        changed = false;
        for x in &af.args {
            if !s.contains(x) && defends(af, &s, x) {
                s.insert(x);
                changed = true;
            }
        }
    }
    s
}

/// {arg: "IN"|"OUT"|"UNDEC"} dérivé de l'extension grounded de `af`.
fn labelling_depuis_grounded(af: &Af) -> BTreeMap<&'static str, &'static str> {
    let ext = grounded(af);
    let mut labels: BTreeMap<&'static str, &'static str> = BTreeMap::new();
    for x in &af.args {
        if ext.contains(x) {
            labels.insert(x, "IN");
        } else if ext.iter().any(|a| af.attacks.contains(&(*a, *x))) {
            labels.insert(x, "OUT");
        } else {
            labels.insert(x, "UNDEC");
        }
    }
    labels
}

/// Étiquetage après chaque arrivée, sous un ordre de délivrance donné.
/// Une attaque ne devient ACTIVE que lorsque ses deux extrémités sont arrivées.
/// (La garde ValueError du Python devient un commentaire : l'invariant — l'ordre
/// énumère exactement les arguments — est garanti par les deux appels ci-dessous.)
fn trajectoire(af: &Af, ordre: &[&'static str]) -> Vec<BTreeMap<&'static str, &'static str>> {
    let mut etapes = Vec::new();
    for k in 1..=ordre.len() {
        let arrives: BTreeSet<&str> = ordre[..k].iter().copied().collect();
        let actives: BTreeSet<(&'static str, &'static str)> = af.attacks.iter()
            .filter(|(a, b)| arrives.contains(a) && arrives.contains(b))
            .copied()
            .collect();
        let sub = Af { args: ordre[..k].to_vec(), attacks: actives };
        etapes.push(labelling_depuis_grounded(&sub));
    }
    etapes
}

fn montre(af: &Af, ordre: &[&'static str]) -> BTreeMap<&'static str, &'static str> {
    println!("Ordre d'arrivée : {}", ordre.join(" -> "));
    let etapes = trajectoire(af, ordre);
    for (i, lab) in etapes.iter().enumerate() {
        let k = i + 1;
        let vue: Vec<String> = ordre[..k].iter().map(|a| format!("{}={}", a, lab[*a])).collect();
        println!("  étape {} : {}", k, vue.join("   "));
    }
    etapes.last().cloned().unwrap_or_default()
}

let af1 = Af {
    args: vec!["c", "b", "a", "d"],
    attacks: [("a", "b"), ("b", "a"), ("c", "c")].iter().copied().collect(),
};
let fin_1 = montre(&af1, &["d", "c", "a", "b"]);
println!();
let fin_2 = montre(&af1, &["a", "b", "c", "d"]);
println!();
let t1 = trajectoire(&af1, &["d", "c", "a", "b"]);
let t2 = trajectoire(&af1, &["a", "b", "c", "d"]);
// BTreeMap == et Vec<BTreeMap> == sont insensibles à l'ordre d'insertion, comme les dict Python.
println!("Étiquetage FINAL identique ? {}", py_bool(fin_1 == fin_2));
println!("Trajectoires identiques    ? {}", py_bool(t1 == t2));


Ordre d'arrivée : d -> c -> a -> b


  étape 1 : d=IN


  étape 2 : d=IN   c=UNDEC


  étape 3 : d=IN   c=UNDEC   a=IN


  étape 4 : d=IN   c=UNDEC   a=UNDEC   b=UNDEC


Ordre d'arrivée : a -> b -> c -> d


... (2 outputs left out)

  étape 3 : a=UNDEC   b=UNDEC   c=UNDEC


  étape 4 : a=UNDEC   b=UNDEC   c=UNDEC   d=IN


Étiquetage FINAL identique ? True


Trajectoires identiques    ? False


### Ce que la trajectoire montre

Les deux ordres convergent vers le **même** étiquetage final — $d$=IN, $c$=UNDEC, $a$ et
$b$=UNDEC — mais par des chemins différents. Dans l'ordre $a \to b \to c \to d$, l'argument
$a$ est **IN** tant qu'il est seul, puis bascule en **UNDEC** dès que son attaquant $b$
arrive : l'acceptation est *révisée*, pas raffinée. C'est la non-monotonie, rendue visible
par le seul fait d'avoir regardé la suite plutôt que sa limite.

Deux remarques pour la suite :

- **$c$ est UNDEC, pas OUT.** L'argument auto-attaqué n'est rejeté par personne — il
  s'exclut lui-même sans qu'un IN ne l'attaque. La règle « OUT $=$ attaqué par un IN »
  ne s'applique donc pas à lui, et c'est bien ce que l'énoncé de l'Exercice 3 anticipe.
- **L'ordre est un paramètre, pas un détail.** Si la trajectoire dépend de l'ordre
  d'arrivée, alors comparer deux protocoles de débat revient à comparer deux trajectoires
  sur le *même* AF — ce que l'étiquetage final, à lui seul, ne permet pas.

### Lien avec le substrat 2025-EPITA-IS

Cette construction existe sous forme outillée dans le dépôt *2025-Epita-Intelligence-Symbolique*,
en `abs_arg_dung/aif_adapter.py` : `aif_labelling_trajectory` construit la même suite
d'étiquetages à partir d'un flux d'arguments et d'attaques **AIF** (undercut / undermine /
rebut), avec `aif_label_transitions` pour extraire les basculements et `render_aif_trajectory`
pour l'affichage. Chaque étape y est vérifiée contre les backends Dung (natif et TweetyProject)
qui se servent mutuellement d'oracle.

Le vocabulaire de sortie est le même à la casse près (`in` / `out` / `undec`), de sorte que
les étiquetages produits ici et là-bas se comparent directement. La cellule ci-dessus reste
volontairement **autonome** — Rust standard (`std`), aucune dépendance — pour que le notebook
s'exécute sans installer quoi que ce soit.

Une conséquence de cette autonomie mérite d'être dite, pour que le motif de démo ne soit
pas pris pour l'algorithme de production : `trajectoire` recalcule `grounded` **de zéro**
à chaque arrivée, et `grounded` énumère les parties de l'ensemble d'arguments — le coût
est donc exponentiel à chaque étape. C'est sans conséquence sur les quatre arguments
canoniques, et rédhibitoire sur un flux réel. Le substrat 2025-EPITA-IS procède par
**propagation incrémentale** des étiquettes : il met à jour l'étiquetage à partir du
précédent au lieu de le recalculer, ce qui est précisément ce que la structure de
trajectoire rend possible.


## À retenir

Ce notebook a reconstruit, de zéro et en Rust standard, les trois sémantiques
fondatrices de l'argumentation abstraite de Dung :

- **Grounded** — l'unique extension complète minimale, attitude sceptique.
- **Preferred** — les extensions complètes maximales, attitude crédule (potentiellement
  multiples et incompatibles).
- **Stable** — les extensions admissibles qui statuent sur tout argument extérieur ;
  **non garantie d'existence**.

Le cas canonique $\langle \{a,b,c,d\}, \{(a,b),(b,a),(c,c)\} \rangle$ a montré les trois
verdicts diverger simultanément : grounded $= \{d\}$, preferred $= \{\{a,d\}, \{b,d\}\}$,
stable inexistante. C'est la richesse structurelle qui justifie l'existence de trois
sémantiques plutôt qu'une seule : aucune ne capte à elle seule toutes les nuances de
l'acceptabilité argumentative.

Le § 4bis complète ce tableau : la **semi-stable** (Caminada 2006) répare le défaut
constaté — elle existe toujours et coïncide avec la stable quand celle-ci existe — et
l'**idéale** (Dung, Mancarella & Toni 2007) fournit le verdict unique sceptique
$\text{grounded} \subseteq \text{idéale} \subseteq \bigcap\text{preferred}$, celui
qu'exige un pipeline qui doit trancher.


## Vérifiez votre compréhension

**Question 1 — Sur l'AF canonique ⟨{a,b,c,d}, {(a,b),(b,a),(c,c)}⟩, pourquoi grounded = {d} alors que preferred contient {a,d} et {b,d} ?**

La première cellule de la section 4 imprime `grounded : {d} (unique, sceptique)` et `preferred : ['{b,d}', '{a,d}'] (2 ext., crédule)`.

<details><summary>Réponse</summary>

Parce que `c` s'auto-attaque (`c->c`) : il ne peut jamais être accepté, donc n'apparaît dans aucune extension. `d` est inattaqué : il entre dans grounded dès la première itération. `a` et `b` se battent mutuellement et rien ne défait l'attaquant de l'un ou de l'autre au départ — donc grounded, qui n'ajoute que le défendu, s'arrête à `{d}`. Les extensions preferred **maximalisent** : chacune prend parti (`{a,d}` ou `{b,d}`), d'où deux verdicts crédules incompatibles. La chaîne imprimée en section 4 (« Lecture du verdict ») le vérifie : `grounded ⊊ preferred : True` et `stable existe ? False`.

</details>

**Question 2 — Pourquoi la semi-stable est-elle appelée « réparation » de la stable ? Que se passe-t-il sur af3 = ⟨{a,b,c}, {a->b}⟩ ?**

La cellule [4bis] imprime `stable : AUCUNE` / `semi-stable : ['{b,d}', '{a,d}']` sur af1, puis sur af3 : `stable : ['{a,c}']`, `semi-stable : ['{a,c}']` et `coïncidence semi-stable == stable (setwise) : True`.

<details><summary>Réponse</summary>

La stable n'existe pas toujours (af1 en est le contre-exemple imprimé) ; la semi-stable (Caminada 2006) existe **toujours** et coïncide avec la stable quand celle-ci existe — c'est le sens exact des deux mesures affichées : réparation sans perte sur les cas où la stable est définie. Sur af3, l'attaque isolée `a->b` rend `{a,c}` stable, et les trois sémantiques (stable, semi-stable, preferred) coïncident : un AF sans cycle n'a pas besoin d'être départagé.

</details>

**Question 3 — L'encadrement g ⊆ i ⊆ ∩preferred est imprimé deux fois, sur deux AF différents : que montre le second témoins (af_id2) que le premier ne montrait pas ?**

La cellule « Lecture des deux témoins » (section 4bis) imprime sur af_id1 `∩ preferred : {x,y}` puis sur af_id2 : `preferred : ['{b,z}', '{a,z}']`, `grounded : {}`, `idéale : {}`, `∩ preferred : {z}` — avec deux fois `encadrement g ⊆ i ⊆ ∩pref : True` et `les trois ne coïncident pas : True`.

<details><summary>Réponse</summary>

Le premier témoins montrait l'encadrement **strict** (trois valeurs distinctes) ; af_id2 montre un cas où grounded et l'idéale **coïncident** (tous deux vides) tout en restant distincts de ∩preferred = {z}. La raison se lit dans les attaques du témoins : `w` (auto-attaqué) menace `z`, et seuls `a` et `b` savent défaire `w` — mais `a` et `b` se battent mutuellement, donc chaque extension preferred n'embarque qu'un des deux combattants. `z` est présent dans chaque extension (d'où `∩ preferred : {z}`), mais l'ensemble `{z}` **seul n'est pas admissible** : `z` ne se défend pas lui-même contre `w`, sa défense exige le combattant, et le combattant dépend du parti pris crédule. L'idéale retient donc ∅. L'encadrement n'est pas toujours strict — c'est pour le mesurer qu'il faut deux témoins.

</details>

**Question 4 — Les deux ordres d'arrivée de la trajectoire d'étiquetage convergent-ils ? Et alors, pourquoi les garder tous les deux ?**

La cellule « trajectoire » (Aller plus loin) imprime les deux suites d'étiquetages puis `Étiquetage FINAL identique ? True` et `Trajectoires identiques ? False`.

<details><summary>Réponse</summary>

Ils convergent vers le **même** étiquetage final (d=IN, c=UNDEC, a et b=UNDEC) — la sémantique est bien définie indépendamment de l'ordre d'évaluation. Mais les trajectoires diffèrent : l'ordre `a -> b -> c -> d` passe a par IN avant de le rétrograder UNDEC, l'ordre inverse n'y passe jamais. La trajectoire n'est pas la sémantique : c'est le chemin computationnel, et sa non-unicité mesurée ici est exactement ce qu'un pipeline agentique incrémental ne doit pas confondre avec le verdict.

</details>

## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place des sémantiques de Dung dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet, du raisonnement formel aux systèmes multi-agents.

**Utiliser le solveur** : [Tweety-5](../Tweety/Tweety-05-Abstract-Argumentation-Python.ipynb) applique ces sémantiques via TweetyProject (JVM) sur des AF plus grands, où l'énumération exhaustive devient prohibitive. Le notebook [2-formal §6](Argumentation-05-Formal-Verification-Python.ipynb) en fait usage dans le pipeline agentique.

**Prouver les théorèmes** : [Tweety-05b-Lean](../Tweety/Tweety-05b-Argumentation-Lean.ipynb) formalise en Lean les inclusions (grounded ⊆ preferred) et les conditions d'existence de la sémantique stable.

**Argumentation structurée** : le formalisme abstrait de Dung ignore le contenu des arguments. ASPIC+, ABA et DeLP (accessibles via Tweety) reconstruisent la relation d'attaque à partir d'une logique sous-jacente — la série [Tweety](../Tweety/) les explore. Les quatre exercices ci-dessus (complétude vs admissibilité, AF stable, étiquetage IN/OUT/UNDEC, axe sceptique complet) prolongent chacun une section du carnet.